# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend songs based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This notebook builds a small local RAG recommender for music preferences.  
The catalog is local, the vector search is local, and OpenAI is only used for embeddings and the final recommendation text.

# Install Dependencies

In [ ]:
# !pip install -qU openai python-dotenv pandas numpy ipython

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Audio, display
from openai import OpenAI

# Load Dataset

For this lab I use a small local music catalog. I also scan the provided `data` folder, so local audio files like `miaow_16k.wav` can be included as extra searchable items.

In [ ]:
# same API key location used in the other labs
env_path = Path(r"E:\AI\Ironhack\Leasons\.env")
local_env_paths = [env_path, Path(".env"), Path("../.env")]

for path in local_env_paths:
    if path.exists():
        load_dotenv(path, override=True)
        break

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found. Check the .env file path.")

client = OpenAI()

# local data folder for this lab
possible_data_dirs = [
    Path(r"E:\AI\Ironhack\Leasons\Week18\lab-music-recommendations-main\lab-music-recommendations-main\data"),
    Path("./data"),
    Path("../data"),
    Path("/mnt/data"),
]

DATA_DIR = next((folder for folder in possible_data_dirs if folder.exists()), possible_data_dirs[0])
DATA_DIR

The next cell checks which audio files are available locally.

In [ ]:
audio_extensions = ["*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a"]
audio_files = []

for extension in audio_extensions:
    audio_files.extend(DATA_DIR.rglob(extension))

audio_files = sorted(audio_files)
print(f"Audio files found: {len(audio_files)}")
for file in audio_files[:10]:
    print(file.name)

Now I create a small music catalog for the recommender. Each row is one song with basic metadata.

In [ ]:
tracks = [
    {
        "title": "Midnight Coffee",
        "artist": "LoFi Room",
        "genre": "lo-fi hip hop",
        "mood": "calm, focused, warm",
        "tempo": "slow",
        "description": "Soft drums, vinyl texture, mellow piano and a relaxed study feeling."
    },
    {
        "title": "Neon Night Drive",
        "artist": "Synth Lane",
        "genre": "synthwave",
        "mood": "energetic, cinematic, night",
        "tempo": "medium-fast",
        "description": "Retro synths, steady electronic drums and a late-night driving atmosphere."
    },
    {
        "title": "Sunny Walk",
        "artist": "The Small Parks",
        "genre": "indie pop",
        "mood": "happy, light, positive",
        "tempo": "medium",
        "description": "Bright guitar, catchy chorus and an easy weekend feeling."
    },
    {
        "title": "Rainy Window",
        "artist": "Piano Notes",
        "genre": "ambient piano",
        "mood": "sad, peaceful, reflective",
        "tempo": "slow",
        "description": "Minimal piano with soft pads, good for reading or relaxing."
    },
    {
        "title": "Desert Road",
        "artist": "Blue Canyon",
        "genre": "blues rock",
        "mood": "raw, warm, road trip",
        "tempo": "medium",
        "description": "Electric guitar, blues rhythm and dusty road trip energy."
    },
    {
        "title": "Deep Focus",
        "artist": "Code Pulse",
        "genre": "minimal techno",
        "mood": "focused, repetitive, clean",
        "tempo": "fast",
        "description": "Clean kick, simple synth loop and a concentration-friendly groove."
    },
    {
        "title": "Island Breeze",
        "artist": "Sea Sound",
        "genre": "reggae pop",
        "mood": "relaxed, sunny, beach",
        "tempo": "medium",
        "description": "Offbeat guitar, soft bass and a chilled beach mood."
    },
    {
        "title": "Morning Run",
        "artist": "Fit Beat",
        "genre": "dance pop",
        "mood": "motivating, bright, workout",
        "tempo": "fast",
        "description": "Strong beat, simple hook and energetic rhythm for running or training."
    },
    {
        "title": "Soft Lights",
        "artist": "Nora Blue",
        "genre": "r&b soul",
        "mood": "smooth, romantic, evening",
        "tempo": "slow-medium",
        "description": "Warm vocals, soft bass and a relaxed night-time feeling."
    },
    {
        "title": "City Jazz",
        "artist": "Metro Trio",
        "genre": "jazz",
        "mood": "classy, relaxed, urban",
        "tempo": "medium",
        "description": "Upright bass, brushed drums and smooth saxophone."
    },
    {
        "title": "Acoustic Sunday",
        "artist": "Maya Fields",
        "genre": "acoustic folk",
        "mood": "soft, honest, peaceful",
        "tempo": "slow-medium",
        "description": "Acoustic guitar, simple vocal melody and a calm Sunday feeling."
    },
    {
        "title": "Arcade Hearts",
        "artist": "Pixel Kids",
        "genre": "electro pop",
        "mood": "fun, playful, colorful",
        "tempo": "fast",
        "description": "Chiptune sounds, pop drums and a playful video-game energy."
    },
]

music_df = pd.DataFrame(tracks)
music_df

If there is any local audio file, I add it as an extra searchable item so the data folder is included.

In [ ]:
extra_rows = []

for file in audio_files:
    extra_rows.append({
        "title": file.stem.replace("_", " ").title(),
        "artist": "Local data folder",
        "genre": "audio sample",
        "mood": "playful, experimental",
        "tempo": "unknown",
        "description": f"Local audio file from the data folder: {file.name}.",
        "path": str(file)
    })

music_df["path"] = ""
if extra_rows:
    music_df = pd.concat([music_df, pd.DataFrame(extra_rows)], ignore_index=True)

music_df.tail()

# Prepare Documents

For the vector search, I combine the important columns into one text field.

In [ ]:
def make_document(row):
    return (
        f"Title: {row['title']}\n"
        f"Artist: {row['artist']}\n"
        f"Genre: {row['genre']}\n"
        f"Mood: {row['mood']}\n"
        f"Tempo: {row['tempo']}\n"
        f"Description: {row['description']}"
    )

music_df["document"] = music_df.apply(make_document, axis=1)
print(music_df.loc[0, "document"])

# Load Embedding Model

The embeddings are created with OpenAI and stored locally in memory for the search step.

In [ ]:
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")


def get_embeddings(texts, batch_size=64):
    all_embeddings = []
    texts = list(texts)

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch
        )
        all_embeddings.extend([item.embedding for item in response.data])

    return np.array(all_embeddings, dtype="float32")


def get_embedding(text):
    return get_embeddings([text])[0]

# Initializing the Local Index

Here I create the local vector index. It is just a NumPy matrix, so we do not need an external vector database for this lab.

In [ ]:
cache_path = DATA_DIR / "music_catalog_embeddings.npy"

if cache_path.exists():
    saved_embeddings = np.load(cache_path)
    if saved_embeddings.shape[0] == len(music_df):
        embeddings = saved_embeddings.astype("float32")
    else:
        embeddings = get_embeddings(music_df["document"].tolist())
        np.save(cache_path, embeddings)
else:
    embeddings = get_embeddings(music_df["document"].tolist())
    try:
        np.save(cache_path, embeddings)
    except Exception as error:
        print("Embedding cache was not saved:", error)

embeddings.shape

In [ ]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(norms, 1e-10)

embedding_index = normalize(embeddings)
print("Local index shape:", embedding_index.shape)

# Querying

This function retrieves the most similar songs for a user preference.

In [ ]:
def search_music(user_preference, top_k=5):
    query_embedding = get_embedding(user_preference)
    query_embedding = query_embedding / max(np.linalg.norm(query_embedding), 1e-10)

    scores = embedding_index @ query_embedding

    results = music_df.copy()
    results["score"] = scores
    results = results.sort_values("score", ascending=False).head(top_k)

    return results[["title", "artist", "genre", "mood", "tempo", "description", "score", "document", "path"]]

In [ ]:
search_music("I want calm music for studying with soft piano or lo-fi beats", top_k=5)[
    ["title", "artist", "genre", "mood", "score"]
]

Now I add the generation part. The LLM only receives the retrieved songs as context.

In [ ]:
def build_context(results):
    lines = []
    for _, row in results.iterrows():
        lines.append(
            f"- {row['title']} by {row['artist']} | "
            f"genre: {row['genre']} | mood: {row['mood']} | "
            f"tempo: {row['tempo']} | score: {row['score']:.3f}\n"
            f"  {row['description']}"
        )
    return "\n".join(lines)

In [ ]:
def recommend_songs(user_preference, top_k=4):
    results = search_music(user_preference, top_k=top_k)
    context = build_context(results)

    prompt = f"""
User preference:
{user_preference}

Songs retrieved from the local catalog:
{context}

Recommend the best songs from the retrieved list.
Keep the answer short and explain why each one fits.
"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0.4,
        messages=[
            {
                "role": "system",
                "content": "You are a music recommendation assistant. Use only the provided catalog results."
            },
            {"role": "user", "content": prompt}
        ]
    )

    print(response.choices[0].message.content)
    return results[["title", "artist", "genre", "mood", "score"]]

## Example 1: study music

In [ ]:
study_results = recommend_songs(
    "I need calm music for studying. I like soft piano, warm lo-fi beats and relaxing sounds.",
    top_k=4
)
study_results

## Example 2: workout music

In [ ]:
workout_results = recommend_songs(
    "Recommend energetic songs for running and gym training. I want something fast and motivating.",
    top_k=4
)
workout_results

## Example 3: evening music

In [ ]:
evening_results = recommend_songs(
    "I want smooth evening music, maybe jazz, r&b or something relaxed for dinner.",
    top_k=4
)
evening_results

# Listen to Local Audio

The uploaded audio file from the data folder can still be played here.

In [ ]:
local_audio = next((file for file in audio_files if file.name == "miaow_16k.wav"), None)

if local_audio is not None:
    display(Audio(str(local_audio)))
else:
    print("miaow_16k.wav was not found in the data folder.")

This query should find the local audio sample because the preference mentions playful animal sounds.

In [ ]:
search_music("I want a playful experimental sound with animal or cat sounds", top_k=5)[
    ["title", "artist", "genre", "mood", "score"]
]

# Similar Songs Helper

This helper searches for songs similar to a song title already in the catalog.

In [ ]:
def find_similar_songs(title, top_k=5):
    matches = music_df[music_df["title"].str.lower() == title.lower()]

    if matches.empty:
        print("Song not found in the catalog.")
        return pd.DataFrame()

    song_text = matches.iloc[0]["document"]
    results = search_music(song_text, top_k=top_k + 1)
    results = results[results["title"].str.lower() != title.lower()].head(top_k)

    return results[["title", "artist", "genre", "mood", "tempo", "score"]]

In [ ]:
find_similar_songs("Midnight Coffee", top_k=5)

These results are mostly calm or focus-related, which makes sense for the query song.

In [ ]:
find_similar_songs("Morning Run", top_k=5)

These results should be more energetic because the source song is a workout/dance pop track.

# Final Test

In [ ]:
my_preference = "I want music for coding at night. It should be electronic, focused and not too distracting."
final_results = recommend_songs(my_preference, top_k=4)
final_results

# Short Conclusion

This is a local RAG recommender because the catalog is stored locally, the retrieval is done with a local NumPy vector index, and the model only uses the retrieved songs to write the recommendation. The system is simple, but it is enough to recommend songs from user preferences using embeddings and retrieval.

# Delete the Index

There is no external index to delete because this lab uses a local vector index.

In [ ]:
# Nothing to delete. The local index is only stored in memory.
print("Local RAG completed.")